# 11 — Iter 2 / Phase A: Stage 3 fix

**Hedef**: MVP Stage 3 LDA %35 / MLP %38'i iyileştirmek. Sebep analizi (sanity check):
- Modeller test bull market'ta sürekli **Sell** tahmin etti (LDA %99.6 Sell, MLP %86.5 Sell)
- Sebep 1: Stage 3 input'unun çoğu **mean-reverting** osilatör (RSI, Stoch, CCI) → bull market'ta sürekli overbought → Sell
- Sebep 2: ±%1 sabit threshold çok dar — BTC 5-day std %7.9, gürültü domine
- Sebep 3: Train period (2014-2024) ayı dönemleri içeriyor → Sell pattern overweight

## Phase A — 3 fix
1. **Adaptive (volatility-adjusted) signal label** primary olarak kullan
2. **Trend-following features** ekle Stage 3 input'una (log returns, above_SMA200, ADX strong-trend flag, donchian, sharpe proxy)
3. **Modeller**: LDA, MLP, XGBoost, LightGBM, RandomForest (5 model)


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

plt.style.use("seaborn-v0_8-darkgrid")
pd.set_option("display.max_columns", 50)

from src.utils.config import cfg
from src.utils.helpers import save_csv, chronological_train_test_split
from src.features.technical_indicators import (
    compute_oscillator_indicators,
    compute_volatility_indicators,
    compute_volume_indicators,
    compute_trend_following_features,
)
from src.models.stage3_trainer import tune_stage3
from src.evaluation.metrics import compute_all_metrics
from src.evaluation.backtester import Backtester

config = cfg()
TEST_SIZE = config["training"]["test_size"]
RANDOM_STATE = config["training"]["random_state"]
STEP_MONTHS = 6
MIN_TRAIN_MONTHS = 12
N_TRIALS = {"lda": 8, "mlp": 6, "xgboost": 12, "lightgbm": 12, "random_forest": 10}

## 1. Stage 3 v2 feature set

v1 (MVP) = oscillator + volatility + volume = 18 feat

v2 (iter 2) = v1 + **trend_following** (10 yeni feat) = 28 feat

In [ ]:
btc = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "btc_aligned.csv", index_col=0, parse_dates=True)
ohlcv = btc[["Open", "High", "Low", "Close", "Volume"]]

osc = compute_oscillator_indicators(ohlcv)
vol = compute_volatility_indicators(ohlcv)
volu = compute_volume_indicators(ohlcv)
trend_follow = compute_trend_following_features(ohlcv)

stage3_v2 = pd.concat([osc, vol, volu, trend_follow], axis=1).dropna()
print(f"Stage 3 v2 features: {stage3_v2.shape}")
print(f"  oscillator     : {osc.shape[1]}")
print(f"  volatility     : {vol.shape[1]}")
print(f"  volume         : {volu.shape[1]}")
print(f"  trend_following: {trend_follow.shape[1]}  (yeni)")
print(f"  TOTAL          : {stage3_v2.shape[1]}")

save_csv(stage3_v2, PROJECT_ROOT / "data" / "processed" / "btc_features_stage3_v2.csv")

## 2. Adaptive signal labels primary

In [ ]:
y_v1 = pd.read_csv(PROJECT_ROOT / "data" / "labels" / "btc_signal_labels_fixed.csv", index_col=0, parse_dates=True).iloc[:, 0]
y_v2 = pd.read_csv(PROJECT_ROOT / "data" / "labels" / "btc_signal_labels_adaptive.csv", index_col=0, parse_dates=True).iloc[:, 0]

comp_dist = pd.DataFrame({
    "v1 fixed +/-1%": (y_v1.value_counts(normalize=True) * 100).round(1),
    "v2 adaptive 0.5xstd": (y_v2.value_counts(normalize=True) * 100).round(1),
})
print(comp_dist)
fig, ax = plt.subplots(figsize=(8, 4))
comp_dist.plot(kind="bar", ax=ax, color=["#3498DB", "#E74C3C"])
ax.set_title("v1 vs v2 signal label distributions")
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "iter2_label_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
stage1_oof = pd.read_csv(PROJECT_ROOT / "data" / "labels" / "btc_stage1_oof_lda.csv", index_col=0, parse_dates=True)
stage2_oof = pd.read_csv(PROJECT_ROOT / "data" / "labels" / "btc_oof_regime_posterior.csv", index_col=0, parse_dates=True)

common = (stage3_v2.index
          .intersection(y_v2.index)
          .intersection(stage1_oof.index)
          .intersection(stage2_oof.index))
X_osc = stage3_v2.loc[common]
y = y_v2.loc[common]
s1_oof = stage1_oof.loc[common]
s2_oof = stage2_oof.loc[common]

X_osc_train, X_osc_test = chronological_train_test_split(X_osc, test_size=TEST_SIZE)
y_train, y_test = y.loc[X_osc_train.index], y.loc[X_osc_test.index]
s1_train, s1_test = s1_oof.loc[X_osc_train.index], s1_oof.loc[X_osc_test.index]
s2_train, s2_test = s2_oof.loc[X_osc_train.index], s2_oof.loc[X_osc_test.index]

print(f"Train: {X_osc_train.shape}  Test: {X_osc_test.shape}")
print(f"Train class dist: {(y_train.value_counts(normalize=True) * 100).round(1).to_dict()}")
print(f"Test  class dist: {(y_test.value_counts(normalize=True) * 100).round(1).to_dict()}")

## 3. 5-model retrain (Optuna walk-forward)

In [ ]:
results = {}
for clf_name in ["lda", "mlp", "xgboost", "lightgbm", "random_forest"]:
    print(f"\n{'='*60}\n{clf_name.upper()}\n{'='*60}")
    t0 = time.time()
    res = tune_stage3(
        X_osc_train, y_train, s1_train, s2_train,
        classifier_name=clf_name,
        n_trials=N_TRIALS[clf_name],
        save_model=False,
        step_months=STEP_MONTHS,
        min_train_months=MIN_TRAIN_MONTHS,
    )
    elapsed = time.time() - t0
    print(f"  best params: {res['best_params']}")
    print(f"  WF f1_macro: {res['avg_f1_macro']:.4f}")
    print(f"  elapsed:     {elapsed:.1f}s")
    results[clf_name] = res

## 4. Test set evaluation

In [ ]:
X_test_combined = pd.concat([X_osc_test, s1_test, s2_test], axis=1)
train_med = pd.concat([X_osc_train, s1_train, s2_train], axis=1).median()
X_test_clean = X_test_combined.fillna(train_med)

test_metrics = {}
test_signals = pd.DataFrame({"y_true": y_test.values}, index=y_test.index)

for clf_name, res in results.items():
    model = res["model"]
    pred = model.predict(X_test_clean)
    proba = model.predict_proba(X_test_clean)
    metrics = compute_all_metrics(y_test.values, pred, y_proba=proba, classes=list(model.classes_))
    test_metrics[clf_name] = metrics
    test_signals[f"{clf_name}_pred"] = pred
    for i, c in enumerate(model.classes_):
        test_signals[f"{clf_name}_proba_{c}"] = proba[:, i]

summary_rows = []
for clf_name in results:
    summary_rows.append({
        "model": clf_name.upper(),
        "WF_f1": results[clf_name]["avg_f1_macro"],
        "test_acc": test_metrics[clf_name]["accuracy"],
        "test_f1": test_metrics[clf_name]["f1_macro"],
        "test_balanced_acc": test_metrics[clf_name]["balanced_accuracy"],
        "test_mcc": test_metrics[clf_name]["mcc"],
    })
summary = pd.DataFrame(summary_rows).round(4).set_index("model")

print("=== Stage 3 v2 — Test set summary ===")
print(summary.to_string())
save_csv(summary, PROJECT_ROOT / "data" / "labels" / "btc_stage3_v2_summary.csv")

print("\n=== Test predictions distribution ===")
for clf_name in results:
    dist = (test_signals[f"{clf_name}_pred"].value_counts(normalize=True) * 100).round(1).to_dict()
    print(f"  {clf_name:14s}: {dist}")
print(f"  {'TRUE':14s}: {(test_signals['y_true'].value_counts(normalize=True) * 100).round(1).to_dict()}")

## 5. v1 vs v2 comparison + best model backtest

In [ ]:
v1 = pd.read_csv(PROJECT_ROOT / "data" / "labels" / "btc_stage3_summary.csv")

best_clf = summary["test_f1"].idxmax().lower()
print(f"Best v2 model by test F1: {best_clf.upper()} (f1={summary.loc[best_clf.upper()]['test_f1']:.4f})")

test_close = btc["Close"].loc[y_test.index]
bt = Backtester()
bh = bt.run_buy_and_hold(test_close)

all_strategies = {}
for clf_name in results:
    sig = test_signals[f"{clf_name}_pred"]
    all_strategies[f"v2 {clf_name.upper()}"] = bt.run(sig, test_close)

v1_signals = pd.read_csv(PROJECT_ROOT / "data" / "labels" / "btc_test_signals.csv", index_col=0, parse_dates=True)
all_strategies["v1 LDA"] = bt.run(v1_signals["lda_pred"], btc["Close"].loc[v1_signals.index])
all_strategies["v1 MLP"] = bt.run(v1_signals["mlp_pred"], btc["Close"].loc[v1_signals.index])

bt_rows = []
for k, v in all_strategies.items():
    bt_rows.append({
        "strategy": k,
        "total_return": v["total_return"],
        "sharpe": v["sharpe_ratio"],
        "max_dd": v["max_drawdown"],
        "n_trades": v["n_trades"],
        "win_rate": v["win_rate"],
    })
bt_rows.append({
    "strategy": "Buy & Hold",
    "total_return": bh.iloc[-1] / bh.iloc[0] - 1,
    "sharpe": (test_close.pct_change().mean() / test_close.pct_change().std() * np.sqrt(252)),
    "max_dd": ((bh - bh.cummax()) / bh.cummax()).min(),
    "n_trades": 1, "win_rate": np.nan,
})
bt_summary = pd.DataFrame(bt_rows).round(4)
print("\n=== Backtest comparison ===")
print(bt_summary.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
color_map = {
    "v1 LDA": "#7F8C8D", "v1 MLP": "#BDC3C7",
    "v2 LDA": "#3498DB", "v2 MLP": "#9B59B6",
    "v2 XGBOOST": "#E67E22", "v2 LIGHTGBM": "#27AE60", "v2 RANDOM_FOREST": "#16A085",
}
for label, result in all_strategies.items():
    ax.plot(result["equity_curve"].index, result["equity_curve"].values,
            label=f"{label} ret={result['total_return']:+.1%} sh={result['sharpe_ratio']:.2f}",
            color=color_map.get(label, "#34495E"),
            linewidth=1.3, alpha=0.85)
ax.plot(bh.index, bh.values, label=f"Buy&Hold ret={bh.iloc[-1]/bh.iloc[0]-1:+.1%}",
        color="#2C3E50", linewidth=1.5, linestyle="--")
ax.set_title("v1 vs v2 strategies — equity curves on test period")
ax.set_ylabel("Portfolio value ($)")
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "iter2_equity_v1_vs_v2.png", dpi=120, bbox_inches="tight")
plt.show()

models_dir = PROJECT_ROOT / "app" / "models"
for clf_name, res in results.items():
    joblib.dump(res["model"], models_dir / f"stage3_{clf_name}_v2.joblib")
save_csv(test_signals, PROJECT_ROOT / "data" / "labels" / "btc_test_signals_v2.csv")
save_csv(bt_summary, PROJECT_ROOT / "data" / "labels" / "btc_backtest_v2_summary.csv")

print("\nv2 artifacts saved.")